In [2]:
# bow vs tfidf

# Import necessary libraries
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os

In [3]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow')
dagshub.init(repo_owner='saurav3k2', repo_name='Mini-mlops-project', mlflow=True)


Accessing as saurav3k2

Initialized MLflow to track repo "saurav3k2/Mini-mlops-project"

Repository saurav3k2/Mini-mlops-project initialized!

In [4]:
# Load the data
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [5]:


def lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    return ''.join([char for char in text if not char.isdigit()])

def lower_case(text):
    return " ".join([word.lower() for word in text.split()])

def removing_punctuations(text):
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def removing_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def normalize_text(df):
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error: {e}')
        raise

In [6]:
# Normalize the text data
df = normalize_text(df)

In [7]:
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

In [8]:
df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


C:\Users\krsor\AppData\Local\Temp\ipykernel_14612\468518138.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


In [9]:
# Set the experiment name
mlflow.set_experiment("BOW vs TF-IDF")


2026/04/09 17:59:05 INFO mlflow.tracking.fluent: Experiment with name 'BOW vs TF-IDF' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/a3f8205b0795468eb427e6df1d8153bc', creation_time=1775737747663, experiment_id='2', last_update_time=1775737747663, lifecycle_stage='active', name='BOW vs TF-IDF', tags={}, trace_location=None, workspace='default'>

In [10]:
# Define feature extraction methods
vectorizers = {
    'BoW': CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}


In [11]:
# Define algorithms
algorithms = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
}


In [12]:
# Start the parent run
with mlflow.start_run(run_name="All Experiments") as parent_run:
    # Loop through algorithms and feature extraction methods (Child Runs)
    for algo_name, algorithm in algorithms.items():
        for vec_name, vectorizer in vectorizers.items():
            with mlflow.start_run(run_name=f"{algo_name} with {vec_name}", nested=True) as child_run:
                X = vectorizer.fit_transform(df['content'])
                y = df['sentiment']
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                

                # Log preprocessing parameters
                mlflow.log_param("vectorizer", vec_name)
                mlflow.log_param("algorithm", algo_name)
                mlflow.log_param("test_size", 0.2)
                
                # Model training
                model = algorithm
                model.fit(X_train, y_train)
                
                # Log model parameters
                if algo_name == 'LogisticRegression':
                    mlflow.log_param("C", model.C)
                elif algo_name == 'MultinomialNB':
                    mlflow.log_param("alpha", model.alpha)
                elif algo_name == 'XGBoost':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                elif algo_name == 'RandomForest':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("max_depth", model.max_depth)
                elif algo_name == 'GradientBoosting':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                    mlflow.log_param("max_depth", model.max_depth)
                
                # Model evaluation
                y_pred = model.predict(X_test)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)
                
                # Log evaluation metrics
                mlflow.log_metric("accuracy", accuracy)
                mlflow.log_metric("precision", precision)
                mlflow.log_metric("recall", recall)
                mlflow.log_metric("f1_score", f1)
                
                # Log model
                mlflow.sklearn.log_model(model, "model")
                
                # Save and log the notebook
                import os
                notebook_path = "exp2_bow_vs_tfidf.ipynb"
                os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
                mlflow.log_artifact(notebook_path)
                # Print the results for verification
                print(f"Algorithm: {algo_name}, Feature Engineering: {vec_name}")
                print(f"Accuracy: {accuracy}")
                print(f"Precision: {precision}")
                print(f"Recall: {recall}")
                print(f"F1 Score: {f1}")

2026/04/09 17:59:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 17:59:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Algorithm: LogisticRegression, Feature Engineering: BoW
Accuracy: 0.7937349397590362
Precision: 0.7846750727449079
Recall: 0.7970443349753694
F1 Score: 0.7908113391984359
🏃 View run LogisticRegression with BoW at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2/runs/fdc44b7bc0f2460dace6127d839cfe49
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2


2026/04/09 18:00:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 18:00:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Algorithm: LogisticRegression, Feature Engineering: TF-IDF
Accuracy: 0.7942168674698795
Precision: 0.777882797731569
Recall: 0.8108374384236453
F1 Score: 0.79401833092137
🏃 View run LogisticRegression with TF-IDF at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2/runs/daa473d0f5ad49fea33052c54ccc1e87
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2


2026/04/09 18:01:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 18:01:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

Algorithm: RandomForest, Feature Engineering: TF-IDF
Accuracy: 0.7667469879518072
Precision: 0.7636544190665343
Recall: 0.7576354679802956
F1 Score: 0.7606330365974283
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2/runs/533293a29dc8411590d6b258f1150491
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2


2026/04/09 18:12:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 18:12:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Algorithm: GradientBoosting, Feature Engineering: BoW
Accuracy: 0.7267469879518073
Precision: 0.8018867924528302
Recall: 0.5862068965517241
F1 Score: 0.6772908366533864
🏃 View run GradientBoosting with BoW at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2/runs/0ec5b82584f54ec8840cd128d1c24f07
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2


2026/04/09 18:14:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 18:14:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Algorithm: GradientBoosting, Feature Engineering: TF-IDF
Accuracy: 0.7243373493975903
Precision: 0.8046767537826685
Recall: 0.5763546798029556
F1 Score: 0.6716417910447762
🏃 View run GradientBoosting with TF-IDF at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2/runs/ca3a486a6c314fd8b0eb39ed206b0133
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2
🏃 View run All Experiments at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2/runs/c899d69c1a9a4f47a2e37a572536674a
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/2
